# 04 · Single Series Forecast with Prediction Intervals

A slightly more realistic walkthrough: a weekly demand series with trend,
yearly seasonality and noise, forecast 52 weeks ahead with **three** confidence
bands (60% and 80%).

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
# Build ~3 years of weekly demand
rng = np.random.default_rng(42)
weeks = np.arange(156)
demand = (
    200
    + 0.4 * weeks                                  # slow growth
    + 40 * np.sin(2 * np.pi * weeks / 52)          # yearly season
    + 15 * np.sin(2 * np.pi * weeks / 13)          # quarterly ripple
    + rng.normal(0, 8, size=weeks.size)
).astype(np.float32)
demand = np.clip(demand, 0, None)
print("weeks of history:", demand.size)

In [ ]:
# Handy constants for indexing the quantile axis
IDX_MEAN = 0
IDX_Q10  = 1   # lower bound, 80% interval
IDX_Q20  = 2   # lower bound, 60% interval
IDX_Q50  = 5   # median
IDX_Q80  = 8   # upper bound, 60% interval
IDX_Q90  = 9   # upper bound, 80% interval

In [ ]:
horizon = 52
point, q = model.forecast(horizon=horizon, inputs=[demand])
point, q = point[0], q[0]   # drop the batch axis for a single series
print("forecast shape:", point.shape, "| quantiles:", q.shape)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hist = demand[-104:]
x_hist = range(len(hist))
x_fc = range(len(hist), len(hist) + horizon)

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.plot(x_hist, hist, color="tab:blue", label="History (2y)")
ax.plot(x_fc, point, color="tab:orange", lw=2, label="Median forecast")
ax.fill_between(x_fc, q[:, IDX_Q10], q[:, IDX_Q90], alpha=0.15,
                color="tab:orange", label="80% interval")
ax.fill_between(x_fc, q[:, IDX_Q20], q[:, IDX_Q80], alpha=0.30,
                color="tab:orange", label="60% interval")
ax.axvline(len(hist) - 0.5, ls="--", color="grey", lw=1)
ax.set_title("52-week demand forecast")
ax.set_xlabel("week"); ax.set_ylabel("units"); ax.legend()
fig.tight_layout(); fig.savefig("weekly_demand_forecast.png", dpi=130)
print("saved weekly_demand_forecast.png")

### Reading the chart
- The **orange line** is your best single-number guess (the median).
- The **shaded bands** show uncertainty — wider = less certain. Use the 80%
  band for planning "worst / best realistic case".